# mb3d in a notebook

3D fractals of Mandelbulb3D, calculated by the program `mb3d` ([mb3d-rs](https://github.com/thr32423/mb3d-rs)) and driven from Python and JavaScript ([mb3d-bindings](https://github.com/thr32423/mb3d-bindings)). EXPERIMENTAL.

Run the cells from the top. The first one fetches what is missing: the modules (this repository) and the program `mb3d` (the latest release, about 100 MB, Linux). Online that is everything; on your own computer it uses the repository the notebook is in, and the program in `MB3D` or on the PATH.

Without a graphics card (as online) the pictures are calculated on the processor: the same pictures, slower.

In [ ]:
import hashlib, os, subprocess, sys, tarfile, urllib.request

REPOSITORY = "https://github.com/thr32423/mb3d-bindings"
RELEASE = "https://github.com/thr32423/mb3d-rs/releases/latest/download/"
ARCHIVE = "mb3d-rs-linux-x64.tar.gz"

# the modules: the repository this notebook is in, else a copy fetched now
if not os.path.exists(os.path.join("..", "python", "mb3d")):
    if not os.path.exists("mb3d-bindings"):
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, "mb3d-bindings"], check=True)
    os.chdir(os.path.join("mb3d-bindings", "notebook"))
sys.path.insert(0, os.path.abspath(os.path.join("..", "python")))


def serves(program):
    """Whether this is a program mb3d that has `serve` (0.3.10 or later)."""
    try:
        answer = subprocess.run([program, "serve"], input=b'{"op":"hello"}\n', capture_output=True, timeout=60)
        return answer.stdout.startswith(b"{")
    except (OSError, subprocess.TimeoutExpired):
        return False


def fetch_program():
    """The latest release of mb3d-rs, checked against its SHA256SUMS and unpacked into the cache folder."""
    folder = os.path.join(os.path.expanduser("~"), ".cache", "mb3d-notebook")
    program = os.path.join(folder, "mb3d")
    if not os.path.exists(program):
        if not (sys.platform == "linux" and os.uname().machine == "x86_64"):
            raise SystemExit("download mb3d from https://github.com/thr32423/mb3d-rs/releases/latest and set MB3D to it")
        os.makedirs(folder, exist_ok=True)
        archive = os.path.join(folder, ARCHIVE)
        urllib.request.urlretrieve(RELEASE + ARCHIVE, archive)
        sums = urllib.request.urlopen(RELEASE + "SHA256SUMS").read().decode()
        wanted = next(line.split()[0] for line in sums.splitlines() if line.endswith(ARCHIVE))
        with open(archive, "rb") as f:
            if hashlib.sha256(f.read()).hexdigest() != wanted:
                raise SystemExit("the downloaded program does not match its checksum")
        with tarfile.open(archive) as t:
            for member in t.getmembers():  # without the archive's top folder
                member.name = member.name.partition("/")[2]
                if member.name:
                    t.extract(member, folder, filter="data")
        os.remove(archive)
    return program


if not serves(os.environ.get("MB3D", "mb3d")):
    os.environ["MB3D"] = fetch_program()
os.environ.setdefault("MB3D", "mb3d")

import mb3d
print(subprocess.run([os.environ["MB3D"], "--version"], capture_output=True, text=True).stdout.strip())

## 1. Without a module: `mb3d serve`

Any language can start `mb3d serve` and keep it running: one JSON request per line into its standard input, one answer per line out of its standard output ([docs/API.md](https://github.com/thr32423/mb3d-rs/blob/master/docs/API.md) lists the requests). Nothing listens on the network.

In [ ]:
import base64, json
from IPython.display import Image, display

serve = subprocess.Popen([os.environ["MB3D"], "serve"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)

def ask(**request):
    serve.stdin.write(json.dumps(request) + "\n")
    serve.stdin.flush()
    return json.loads(serve.stdout.readline())

print(ask(op="hello"))
box = ask(op="open", preset="Amazing Box")
print({key: box[key] for key in ("ok", "scene", "width", "height")})
ask(op="set", scene=box["scene"], changes="zoom = 0.3\niterations = 12")
answer = ask(op="render", scene=box["scene"], width=480)
serve.stdin.close()  # the program ends when its input is closed

print({key: answer[key] for key in ("ok", "width", "height", "seconds", "device")})
Image(base64.b64decode(answer["png"]))

## 2. Python

The module `mb3d` does the same talking. A formula by name comes with a camera that shows it; `engine.open("file.m3p")` takes a parameter file.

In [ ]:
import mb3d

with mb3d.Engine() as engine:
    scene = engine.preset("Amazing Box")          # or engine.open("file.m3p")
    scene.set(zoom=0.3, iterations=12)
    picture = scene.render(width=640, file="box.png")

print(picture)
Image("box.png")

## 3. JavaScript

The same with the module for Node.js. The cell below is a JavaScript program: the notebook hands it to `node`.

In [ ]:
%%script node --input-type=module
import { Engine } from "../js/mb3d.mjs";

const engine = await Engine.start();
const scene = await engine.preset("Amazing Box");  // or engine.open("file.m3p")
await scene.set({ zoom: 0.3, iterations: 12, rotation: [25, -40, 0] });
const picture = await scene.render({ width: 640, file: "box-js.png" });
engine.close();

console.log(`${picture.width} x ${picture.height} in ${picture.seconds} s: ${picture.device}`);

In [ ]:
Image("box-js.png")

## Changing the scene

From here on in Python, with one engine kept running: every picture after the first starts at once. A rendered picture shows itself.

`key = value` settings: the view, the detail, colours, fog, ... `engine.guide()` lists what the keys do.

In [ ]:
engine = mb3d.Engine()
scene = engine.preset("Amazing Box")
scene.set(zoom=0.3, rotation=(25, -40, 0), iterations=14)
scene.render(width=640)

In [ ]:
print(engine.guide()[:1500])

A formula's own values reshape the fractal. `scene.formula_values` shows them; `engine.formulas()` names every formula.

In [ ]:
print(len(engine.formulas()), "formulas")
scene.formula_values

In [ ]:
slot = scene.formula_values[0]["slot"]
scene.set_formula(slot, {"Scale": 2.4})
scene.render(width=640)

## Sliders

Turn, zoom and reshape. The scene keeps what the sliders set.

In [ ]:
from mb3d.notebook import explore

bulb = engine.preset("Integer Power")
explore(bulb)

## Pixels and depth as arrays

For numpy, Pillow, matplotlib: every picture comes with its depth (65535 nearest, 0 background).

In [ ]:
import numpy as np

p = bulb.render(width=480, pixels=True, depth=True)
image = np.frombuffer(p.rgb, np.uint8).reshape(p.height, p.width, 3)
depth = np.frombuffer(p.depth, "<u2").reshape(p.height, p.width)
image.shape, depth.shape, float((depth > 0).mean())

## A sweep

One formula value in eight steps.

In [ ]:
sweep = engine.preset("Amazing Box")
slot = sweep.formula_values[0]["slot"]
for scale in np.linspace(1.6, 3.0, 8):
    sweep.set_formula(slot, {"Scale": float(scale)})
    display(Image(sweep.render(width=240).png))

## Your own parameter files

`engine.open("file.m3p")` takes Mandelbulb3D's `.m3p` and `.m3i` files and text parameters (online: upload the file into the notebook's files first); `scene.save("new.m3p")` writes one.

In [ ]:
scene.save("box.m3p")
again = engine.open("box.m3p")
again.render(width=320)

In [ ]:
engine.close()